In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("segmentation_models_pytorch") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"],
        check=True,
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.4 MB/s eta 0:00:00


In [2]:
import json
import os
import random
from functools import lru_cache

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from pycocotools import mask as mask_utils
from scipy import ndimage
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, Sampler, DataLoader
import time
import segmentation_models_pytorch as smp

In [3]:
INPUT_ROOT = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026" 
OUTPUT_ROOT = "/kaggle/working"

CONFIG = {
    "train_images_dir": f"{INPUT_ROOT}/train/train_images",
    "train_annotations": f"{INPUT_ROOT}/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json",
    "test_images_dir": f"{INPUT_ROOT}/test/test_images",
    "checkpoint_path": f"{OUTPUT_ROOT}/model.pt",
    "checkpoint_model": f"/kaggle/input/models/kumail420/segres-epochs-1-5/pytorch/default/2/model.pt",
    "submission_path": f"{OUTPUT_ROOT}/submission.csv",
    "image_height": 2048,
    "image_width": 2048,
    "tile_size": 512,
    "overlap": 64,
    "val_fraction": 0.15,
    "batch_size": 8,
    "num_workers": 2,
    "epochs": 15,
    "lr": 1e-4,
    "seed": 42,
    "pred_threshold": 0.5,
    "min_area": 200,
    "pos_weight": 200,
    "lr_patience": 3,
    "early_stop_patience": 6,
    "use_amp": True,
    "resume_epoch": 0
}

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(f"{OUTPUT_ROOT}/train.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("filament")


#### Tiling

In [4]:
def _axis_coordinates(size, tile_size, stride):
    """
    Top-left coordinates along one axis, evenly spaced to exactly cover
    [0, size) with `tile_size`-wide tiles.
    A naive fixed-stride walk that clamps the last tile to fit inside
    """
    if size <= tile_size:
        return [0]

    n_tiles = int(np.ceil((size - tile_size) / stride)) + 1
    positions = np.linspace(0, size - tile_size, n_tiles)
    # round + dedup: linspace can produce repeats when n_tiles is large
    # relative to the span, though not for any tile/overlap combo used here
    return sorted({int(round(p)) for p in positions})


def get_tile_coordinates(height, width, tile_size=512, overlap=64):
    """
    Compute top-left (y, x) coordinates for tiles covering the full image,
    with some overlap so filaments crossing tile borders aren't cut cleanly
    Returns a list of (y, x) tuples.
    """
    stride = tile_size - overlap
    ys = _axis_coordinates(height, tile_size, stride)
    xs = _axis_coordinates(width, tile_size, stride)
    return [(y, x) for y in ys for x in xs]


def extract_tile(array, y, x, tile_size=512):
    """Extract a single tile from a 2D array (image or mask).
    np.ascontiguousarray is important here: slicing alone returns a view,
    and OpenCV-based operations (used internally by Albumentations) can
    fail silently on non-contiguous arrays.
    """
    tile = array[y:y + tile_size, x:x + tile_size]
    return np.ascontiguousarray(tile)


def stitch_predictions(pred_tiles, coords, height, width, tile_size=512):
    """
    Reassemble predicted tiles into a full-resolution mask.
    pred_tiles: list of 2D numpy arrays (model output per tile), same order as coords
    coords: list of (y, x) tuples, matching get_tile_coordinates output
    height, width: full image dimensions
    """
    full_pred = np.zeros((height, width), dtype=np.float32)
    count_map = np.zeros((height, width), dtype=np.float32)

    for pred_tile, (y, x) in zip(pred_tiles, coords):
        full_pred[y:y + tile_size, x:x + tile_size] += pred_tile
        count_map[y:y + tile_size, x:x + tile_size] += 1.0

    # avoid division by zero, though every pixel should be covered at least once
    count_map[count_map == 0] = 1.0
    full_pred = full_pred / count_map

    return full_pred

#### Annotation decoding

In [5]:
def load_annotations(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    return data


def build_lookup_tables(data):
    """Build fast lookup dicts for images and annotations by image_id."""
    images_by_id = {img["id"]: img for img in data["images"]}

    anns_by_image_id = {}
    for ann in data["annotations"]:
        anns_by_image_id.setdefault(ann["image_id"], []).append(ann)

    return images_by_id, anns_by_image_id


def polygon_to_mask(segmentation, height, width):
    """
    Convert a single polygon segmentation to a binary mask.
    """
    rle = mask_utils.frPyObjects(segmentation, height, width)

    # merge in case frPyObjects returns multiple RLEs (e.g. multi-part polygon)
    if isinstance(rle, list):
        rle = mask_utils.merge(rle)

    binary_mask = mask_utils.decode(rle)
    return binary_mask


def build_file_to_annotations(data):
    """
    Map file_name -> annotations from *every* annotator who labeled that file.

    The same image appears once per annotator: 1154 image entries cover only
    707 distinct files, and image ids are "<annotator>-<file>" (010101,
    010102, 010103). Keying file_name -> a single image_id discarded 447 of
    the 1154 entries and 2948 of the 8199 annotations, so filaments a second
    annotator marked were fed to the model as background.
    """
    images_by_id, anns_by_image_id = build_lookup_tables(data)

    file_to_anns = {}
    for image_id, image_info in sorted(images_by_id.items()):
        file_name = image_info["file_name"]
        file_to_anns.setdefault(file_name, []).extend(anns_by_image_id.get(image_id, []))

    return file_to_anns


# 1=Left, 2=Right, 3=Unidentifiable. All three mark a filament; only the
# chirality label differs, and the mask is binary filament/background, so
# excluding 3 taught the model to call 3074 of 8199 annotated filaments
# background. Category 4 (Ambiguous) is declared but has zero instances.
DEFAULT_CATEGORY_IDS = frozenset({1, 2, 3})


def build_combined_mask(annotations, height, width, category_ids=DEFAULT_CATEGORY_IDS):
    """
    Combine filament masks for one image into a single binary mask.

    Annotators overlap only partially (pairwise IoU 0.25-0.57 on the files
    with more than one), so this is a union, not a consensus: a pixel any
    annotator called filament counts as foreground.
    """
    combined = np.zeros((height, width), dtype=np.uint8)

    for ann in annotations:
        if category_ids is not None and ann["category_id"] not in category_ids:
            continue
        seg = ann["segmentation"]
        m = polygon_to_mask(seg, height, width)
        combined = np.logical_or(combined, m).astype(np.uint8)

    return combined

#### Dataset + sampler

In [6]:
class SolarFilamentDataset(Dataset):
    def __init__(
        self,
        file_names,
        images_dir,
        annotations_json=None,
        transform=None,
        is_test=False,
        tile_size=512,
        overlap=64,
        image_height=2048,
        image_width=2048,
        mask_cache_size=16,
        return_meta=False,
    ):
        """
        file_names: list of base file names, e.g. "20260901165702Bh.jpeg"
        images_dir: directory containing the raw grayscale images
        annotations_json: path to the COCO-style annotation file (None if is_test=True).
            Masks are decoded from polygon segmentations on the fly, per tile,
            instead of being read from precomputed mask PNGs.
        transform: an Albumentations transform, applied jointly to image and mask tile
        is_test: if True, tiles are still built (for full-image inference later),
                 but no mask is loaded or returned
        tile_size, overlap: passed to get_tile_coordinates
        image_height, image_width: expected full image dimensions (2048x2048 here)
        mask_cache_size: number of full-image masks to keep decoded in memory,
            so the 512x512 tiles of the same image don't each re-run RLE decode
        return_meta: if True (and is_test=False), also return file_name, y, x
            alongside image/mask tiles, so tiles can be stitched back into
            full images for image-level validation metrics
        """
        self.images_dir = images_dir
        self.transform = transform
        self.is_test = is_test
        self.return_meta = return_meta
        self.tile_size = tile_size
        self.image_height = image_height
        self.image_width = image_width

        if not is_test:
            if annotations_json is None:
                raise ValueError("annotations_json is required when is_test=False")
            data = load_annotations(annotations_json)
            self.file_to_anns = build_file_to_annotations(data)
        else:
            self.file_to_anns = None

        # precompute tile coordinates once, shared across all images
        self.tile_coords = get_tile_coordinates(image_height, image_width, tile_size, overlap)

        # build a flat index: one entry per (file_name, tile_coord) pair
        self.index = []
        for file_name in file_names:
            for (y, x) in self.tile_coords:
                self.index.append((file_name, y, x))

        # cache image+mask together per file, so the tile_size**2/overlap tiles
        self._load_file = lru_cache(maxsize=mask_cache_size)(self._load_file_uncached)

    def __len__(self):
        return len(self.index)

    def _load_image(self, file_name):
        """Load image as single-channel grayscale, kept as (H, W)."""
        image_path = os.path.join(self.images_dir, file_name)
        image = Image.open(image_path).convert("L")  # force grayscale, not RGB
        return np.array(image)

    def _build_mask_uncached(self, file_name):
        """Decode the full-image binary mask from polygon annotations."""
        anns = self.file_to_anns.get(file_name, [])
        if not anns:
            return np.zeros((self.image_height, self.image_width), dtype=np.uint8)
        return build_combined_mask(anns, self.image_height, self.image_width)

    def _load_file_uncached(self, file_name):
        """Decode image (and mask, if labeled) for a file once, cached per file."""
        image = self._load_image(file_name)
        mask = None if self.is_test else self._build_mask_uncached(file_name)
        return image, mask

    def __getitem__(self, idx):
        file_name, y, x = self.index[idx]

        image, full_mask = self._load_file(file_name)
        image_tile = extract_tile(image, y, x, self.tile_size)

        if self.is_test:
            if self.transform:
                augmented = self.transform(image=image_tile)
                image_tile = augmented["image"]
            # y, x returned too, so predictions can be stitched back later
            return image_tile, file_name, y, x

        mask_tile = extract_tile(full_mask, y, x, self.tile_size)

        if self.transform:
            # same transform applied to both, so they stay aligned
            augmented = self.transform(image=image_tile, mask=mask_tile)
            image_tile = augmented["image"]
            mask_tile = augmented["mask"]

        # mask stays float, single channel, shape (1, H, W) expected by loss functions
        mask_tile = mask_tile.unsqueeze(0).float()

        if self.return_meta:
            return image_tile, mask_tile, file_name, y, x

        return image_tile, mask_tile


class FileGroupedSampler(Sampler):
    """
    Keeps accesses to the same file clustered together
    """
    def __init__(self, dataset, seed=0):
        self.tiles_per_file = len(dataset.tile_coords)
        assert len(dataset.index) % self.tiles_per_file == 0
        self.num_files = len(dataset.index) // self.tiles_per_file
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        file_order = rng.permutation(self.num_files)
        for f in file_order:
            block = np.arange(f * self.tiles_per_file, (f + 1) * self.tiles_per_file)
            rng.shuffle(block)
            yield from block.tolist()

    def __len__(self):
        return self.tiles_per_file * self.num_files

#### Transforms 

In [7]:
IMG_SIZE = 512

# Single-channel, but the resnet34 encoder is ImageNet-pretrained, and smp
# adapts it to in_channels=1 by summing the RGB conv weights — i.e. the
# encoder still expects ImageNet-normalized input. These are the ImageNet
# RGB stats collapsed to luminance (0.299R + 0.587G + 0.114B), so the input
# distribution matches what the pretrained weights were trained on. Plain
# 0.5/0.5 shifted and scaled it away from that.

GRAYSCALE_MEAN = (0.449,)
GRAYSCALE_STD = (0.226,)


def get_train_transform():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Normalize(mean=GRAYSCALE_MEAN, std=GRAYSCALE_STD),
        ToTensorV2(),
    ])


def get_val_transform():
    return A.Compose([
        A.Normalize(mean=GRAYSCALE_MEAN, std=GRAYSCALE_STD),
        ToTensorV2(),
    ])

#### Model

In [8]:
def build_model(encoder_weights="imagenet"):
    """
    encoder_weights: "imagenet" downloads pretrained weights (training path).
        Pass None on the inference path — the checkpoint overwrites these
        weights anyway, and the download hard-fails in a no-internet Kaggle
        inference kernel.
    """
    model_class = smp.UnetPlusPlus

    model = model_class(
        encoder_name="resnet34",
        encoder_weights=encoder_weights,
        in_channels=1,
        classes=1,
    )

    return model

#### Losses

In [9]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        """
        preds: raw model output (logits), shape (B, 1, H, W)
        targets: ground-truth mask, values 0 or 1, shape (B, 1, H, W)

        The .float() casts are load-bearing under AMP: inside torch.autocast
        these tensors arrive as fp16, and summing a 512x512 tile (262144
        pixels) overflows fp16's 65504 max as soon as the mean sigmoid output
        exceeds ~0.25. The overflowed union pins dice_loss at 1.0 and sends
        NaN gradients back, which GradScaler then skips — every step, forever,
        with no error raised.
        """
        preds = torch.sigmoid(preds.float())
        targets = targets.float()

        preds = preds.view(preds.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (preds * targets).sum(dim=1)
        union = preds.sum(dim=1) + targets.sum(dim=1)

        dice_score = (2.0 * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1.0 - dice_score

        return dice_loss.mean()


class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1.0, pos_weight=None):
        """
        pos_weight: weight on the positive-class BCE term, to counter
        filament pixels being ~1:546 rare against background. Without it,
        plain BCE at 0.5 pushes the model toward predicting all-background.
        Registered as a buffer so it moves with the loss module's `.to(device)`.
        """
        super().__init__()
        self.dice_loss = DiceLoss(smooth=smooth)
        if pos_weight is not None:
            self.register_buffer("pos_weight", torch.tensor(float(pos_weight)))
        else:
            self.pos_weight = None
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight

    def forward(self, preds, targets):
        d_loss = self.dice_loss(preds, targets)
        b_loss = F.binary_cross_entropy_with_logits(preds, targets, pos_weight=self.pos_weight)
        return self.dice_weight * d_loss + self.bce_weight * b_loss

#### Predict-side helpers defined before

In [10]:
def load_model(checkpoint_path, device):
    model = build_model(encoder_weights=None)
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.to(device)
    model.eval()
    return model

def predict_tiles(model, loader, device):
    """
    Run the model on every test tile, and collect predictions grouped
    by file_name, so they can be stitched back together per image.

    Returns: dict {file_name: {"tiles": [...], "coords": [...]}}
    """
    results = {}

    with torch.no_grad():
        for images, file_names, ys, xs in loader:
            images = images.to(device)
            preds = model(images)
            preds = torch.sigmoid(preds).cpu().numpy()  # (B, 1, H, W)

            for i in range(len(file_names)):
                file_name = file_names[i]
                y = ys[i].item()
                x = xs[i].item()
                pred_tile = preds[i, 0]  # (H, W)

                if file_name not in results:
                    results[file_name] = {"tiles": [], "coords": []}

                results[file_name]["tiles"].append(pred_tile)
                results[file_name]["coords"].append((y, x))

    return results


def label_and_filter(binary_mask, min_area=1):
    """
    Label connected components once and drop tiny specks by area (helps with
    the fragmentation penalty mentioned in the evaluation rubric), returning
    both the cleaned binary mask and one instance mask per surviving
    component. Replaces separate clean_mask() + split_into_instances() calls,
    which used to label the same mask twice and, in clean_mask's case, did a
    full-image boolean compare per component instead of a single bincount.

    Returns: (cleaned_binary_mask, [instance_mask, ...])
    """
    labeled, num_features = ndimage.label(binary_mask)
    if num_features == 0:
        return np.zeros_like(binary_mask, dtype=np.uint8), []

    areas = np.bincount(labeled.ravel(), minlength=num_features + 1)
    keep_ids = np.nonzero(areas[1:] >= min_area)[0] + 1

    cleaned = np.isin(labeled, keep_ids).astype(np.uint8)
    instances = [(labeled == label_id).astype(np.uint8) for label_id in keep_ids]

    return cleaned, instances


def mask_to_rle_string(binary_mask):
    """Convert a binary mask to an RLE counts string, per the submission format."""
    rle = mask_utils.encode(np.asfortranarray(binary_mask))
    counts = rle["counts"]
    if isinstance(counts, bytes):
        counts = counts.decode("utf-8")
    return counts


def build_submission(results, threshold, min_area, output_csv, tile_size, image_height, image_width):
    rows = []

    for file_name, data in results.items():
        pred_tiles = data["tiles"]
        coords = data["coords"]

        stitched = stitch_predictions(pred_tiles, coords, image_height, image_width, tile_size)
        binary_mask = (stitched > threshold).astype(np.uint8)
        _, instances = label_and_filter(binary_mask, min_area=min_area)

        base_name = os.path.splitext(file_name)[0]
        for idx, instance_mask in enumerate(instances, start=1):
            filament_id = f"{base_name}_{idx}"
            rle_string = mask_to_rle_string(instance_mask)
            rows.append({"filament_id": filament_id, "segmentation_rle": rle_string})

        # every test image needs at least one row in the submission, even
        # when no filament survives the area filter, so an empty prediction
        # doesn't just drop the image's id from the csv entirely
        if not instances:
            empty_mask = np.zeros((image_height, image_width), dtype=np.uint8)
            rows.append({
                "filament_id": f"{base_name}_1",
                "segmentation_rle": mask_to_rle_string(empty_mask),
            })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Saved submission with {len(df)} rows to {output_csv}")

#### Train-side helpers

In [11]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_one_epoch(model, loader, optimizer, loss_fn, device, scaler=None, log_every_sec=15):
    """
    scaler: a torch.amp.GradScaler. Pass one with enabled=True (only
    meaningful on CUDA) to train under autocast + mixed precision; pass one
    with enabled=False (or None) to train in plain fp32.
    """
    model.train()
    running_loss = 0.0
    seen = 0
    amp_enabled = scaler is not None and scaler.is_enabled()

    total_steps = len(loader)
    start = last_log = time.monotonic()

    for step, (images, masks) in enumerate(loader, start=1):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            preds = model(images)
            loss = loss_fn(preds, masks)

        if amp_enabled:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        seen += images.size(0)

        now = time.monotonic()
        if now - last_log >= log_every_sec or step == total_steps:
            elapsed = now - start
            rate = step / elapsed
            eta = (total_steps - step) / rate if rate > 0 else 0
            log.info(
                f"train step {step}/{total_steps} ({100 * step / total_steps:.0f}%) "
                f"loss={running_loss / seen:.4f} elapsed={elapsed:.0f}s eta={eta:.0f}s"
            )
            last_log = now

    # divide by samples actually seen, not len(dataset): drop_last=True can
    # discard up to batch_size - 1 samples, which never reach the numerator
    return running_loss / seen if seen else 0.0


def _instance_match_counts(pred_instances, gt_instances, iou_thresh):
    """Greedy IoU matching between predicted and ground-truth instances.

    Returns (tp, fp, fn) counts for this one image.
    """
    matched_gt = set()
    tp = 0
    for p_inst in pred_instances:
        best_iou, best_j = 0.0, -1
        for j, g_inst in enumerate(gt_instances):
            if j in matched_gt:
                continue
            intersection = np.logical_and(p_inst, g_inst).sum()
            union = np.logical_or(p_inst, g_inst).sum()
            iou = intersection / union if union > 0 else 0.0
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thresh:
            tp += 1
            matched_gt.add(best_j)

    fp = len(pred_instances) - tp
    fn = len(gt_instances) - len(matched_gt)
    return tp, fp, fn


class _ImageScoreAccumulator:
    """Running tally of image-level validation metrics.

    Each image is scored and discarded as soon as all of its tiles have
    arrived, so only one image's tiles are ever held (see validate()).
    """

    def __init__(self, dataset, threshold, min_area, iou_thresh):
        self.dataset = dataset
        self.threshold = threshold
        self.min_area = min_area
        self.iou_thresh = iou_thresh

        self.dices = []  # only images with at least one ground-truth filament pixel
        self.empty_total = 0
        self.empty_correct = 0
        self.tp = self.fp = self.fn = 0

    def add_image(self, entry):
        dataset = self.dataset
        stitched_pred = stitch_predictions(
            entry["pred_tiles"], entry["coords"], dataset.image_height, dataset.image_width, dataset.tile_size
        )
        stitched_gt = stitch_predictions(
            entry["gt_tiles"], entry["coords"], dataset.image_height, dataset.image_width, dataset.tile_size
        )

        pred_binary, pred_instances = label_and_filter(
            (stitched_pred > self.threshold).astype(np.uint8), min_area=self.min_area
        )
        gt_binary, gt_instances = label_and_filter((stitched_gt > 0.5).astype(np.uint8), min_area=1)

        if gt_binary.sum() == 0:
            self.empty_total += 1
            if pred_binary.sum() == 0:
                self.empty_correct += 1
        else:
            intersection = np.logical_and(pred_binary, gt_binary).sum()
            union = pred_binary.sum() + gt_binary.sum()
            self.dices.append(2.0 * intersection / union if union > 0 else 1.0)

        img_tp, img_fp, img_fn = _instance_match_counts(pred_instances, gt_instances, self.iou_thresh)
        self.tp += img_tp
        self.fp += img_fp
        self.fn += img_fn


def validate(model, loader, loss_fn, device, threshold=0.5, min_area=20, iou_thresh=0.5, log_every_sec=15):
    """
    Runs validation loss per-tile (cheap, matches training objective), but
    computes dice and instance metrics on full stitched images, matching how
    the competition actually scores predictions.

    `loader` must be built with SolarFilamentDataset(..., return_meta=True)
    so tiles carry (file_name, y, x) for stitching, and with shuffle=False so
    a file's tiles arrive contiguously.

    Per-tile dice with additive smoothing scores an empty tile 1.0 regardless
    of the prediction (72.8% of tiles have no filament), so an all-background
    model floors near 0.73 and checkpoint selection rewards collapsing to
    background. Stitching first and reporting empty/non-empty images
    separately avoids that: dice is only computed where there's a filament to
    find, and an all-background model scores 0 there instead of ~0.73.

    Tiles are scored and freed per image rather than buffered for the whole
    val set: holding every prediction+ground-truth pair to the end costs
    ~52 MB/file (~5.6 GB over a 107-file val split), which OOMs constrained
    machines once DataLoader prefetch is stacked on top. Predictions are kept
    as float16 and ground truth as uint8 in the buffer for the same reason.
    """
    dataset = loader.dataset
    tiles_per_file = len(dataset.tile_coords)
    model.eval()
    running_loss = 0.0
    seen = 0

    acc = _ImageScoreAccumulator(dataset, threshold, min_area, iou_thresh)
    pending = {}  # file_name -> {"pred_tiles": [...], "gt_tiles": [...], "coords": [...]}

    total_steps = len(loader)
    start = last_log = time.monotonic()

    with torch.no_grad():
        for step, (images, masks, file_names, ys, xs) in enumerate(loader, start=1):
            images = images.to(device)
            masks = masks.to(device)

            preds = model(images)
            loss = loss_fn(preds, masks)
            running_loss += loss.item() * images.size(0)
            seen += images.size(0)

            now = time.monotonic()
            if now - last_log >= log_every_sec or step == total_steps:
                log.info(f"val step {step}/{total_steps} ({100 * step / total_steps:.0f}%) loss={running_loss / seen:.4f}")
                last_log = now

            probs = torch.sigmoid(preds).cpu().numpy()[:, 0].astype(np.float16)  # (B, H, W)
            gt = masks.cpu().numpy()[:, 0].astype(np.uint8)  # (B, H, W)

            for i, file_name in enumerate(file_names):
                entry = pending.setdefault(file_name, {"pred_tiles": [], "gt_tiles": [], "coords": []})
                entry["pred_tiles"].append(probs[i])
                entry["gt_tiles"].append(gt[i])
                entry["coords"].append((ys[i].item(), xs[i].item()))

                if len(entry["coords"]) == tiles_per_file:
                    acc.add_image(entry)
                    del pending[file_name]

    # nothing should be left: every file contributes exactly tiles_per_file
    # tiles and val_loader has no drop_last. Score any stragglers anyway
    # rather than silently dropping them from the metrics.
    for entry in pending.values():
        acc.add_image(entry)

    avg_loss = running_loss / seen if seen else 0.0

    dices = acc.dices
    empty_total = acc.empty_total
    empty_correct = acc.empty_correct
    tp, fp, fn = acc.tp, acc.fp, acc.fn

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    instance_f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    metrics = {
        "dice": float(np.mean(dices)) if dices else 0.0,
        "n_nonempty_images": len(dices),
        "n_empty_images": empty_total,
        "empty_correct_frac": empty_correct / empty_total if empty_total > 0 else 1.0,
        "instance_precision": precision,
        "instance_recall": recall,
        "instance_f1": instance_f1,
    }
    return avg_loss, metrics

#### Drivers

In [12]:
def list_image_files(images_dir):
    return sorted(f for f in os.listdir(images_dir) if f.lower().endswith((".jpeg", ".jpg", ".png")))


def build_loaders(cfg):
    file_names = list_image_files(cfg["train_images_dir"])
    train_files, val_files = train_test_split(
        file_names, test_size=cfg["val_fraction"], random_state=cfg["seed"]
    )

    common_kwargs = dict(
        images_dir=cfg["train_images_dir"],
        annotations_json=cfg["train_annotations"],
        tile_size=cfg["tile_size"],
        overlap=cfg["overlap"],
        image_height=cfg["image_height"],
        image_width=cfg["image_width"],
    )

    train_dataset = SolarFilamentDataset(
        train_files, transform=get_train_transform(), **common_kwargs
    )
    val_dataset = SolarFilamentDataset(
        val_files, transform=get_val_transform(), return_meta=True, **common_kwargs
    )

    # shuffle=True at the DataLoader level shuffles individual tiles, defeating
    # the dataset's per-file image/mask cache (see SolarFilamentDataset above).
    # Shuffle file order instead, keeping a file's tiles clustered together.
    train_sampler = FileGroupedSampler(train_dataset, seed=cfg["seed"])
    train_loader = DataLoader(
        train_dataset, batch_size=cfg["batch_size"], sampler=train_sampler,
        num_workers=cfg["num_workers"], pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg["batch_size"], shuffle=False,
        num_workers=cfg["num_workers"], pin_memory=True,
    )
    return train_loader, val_loader, train_sampler


def train(cfg, device, resume=False):
    train_loader, val_loader, train_sampler = build_loaders(cfg)

    model = build_model().to(device)
    loss_fn = DiceBCELoss(pos_weight=cfg["pos_weight"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=cfg["lr_patience"]
    )
    amp_enabled = cfg["use_amp"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=amp_enabled)

    best_dice = -1.0
    start_epoch = 1

    if resume:
        ckpt = torch.load(cfg["checkpoint_model"], map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        best_dice = ckpt["best_dice"]
        start_epoch = cfg["resume_epoch"]
        epochs_without_improvement = 0
        print(f"resumed, best_dice={best_dice:.4f}")
    else:
        epochs_without_improvement = 0
        print(f"resume not requested and no checkpoint at {cfg['checkpoint_model']}, starting fresh")


    os.makedirs(os.path.dirname(cfg["checkpoint_path"]), exist_ok=True)
    epochs_without_improvement = 0

    for epoch in range(start_epoch, start_epoch + cfg["epochs"]):
        train_sampler.set_epoch(epoch)
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device, scaler=scaler)
        val_loss, metrics = validate(
            model, val_loader, loss_fn, device,
            threshold=cfg["pred_threshold"], min_area=cfg["min_area"],
        )
        scheduler.step(metrics["dice"])

        print(
            f"epoch {epoch}/{cfg['epochs']} "
            f"lr={optimizer.param_groups[0]['lr']:.2e} "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
            f"dice={metrics['dice']:.4f} (n={metrics['n_nonempty_images']}) "
            f"empty_correct={metrics['empty_correct_frac']:.4f} (n={metrics['n_empty_images']}) "
            f"instance_f1={metrics['instance_f1']:.4f} "
            f"(precision={metrics['instance_precision']:.4f} recall={metrics['instance_recall']:.4f})"
        )

        # checkpoint on full-image dice over images that actually contain a
        # filament, not per-tile dice inflated by empty-tile smoothing
        if metrics["dice"] > best_dice:
            best_dice = metrics["dice"]
            epochs_without_improvement = 0
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "best_dice": best_dice,
                "epoch": epoch,
                "epochs_without_improvement": epochs_without_improvement,
            }, cfg["checkpoint_path"])
            print(f"  saved new best checkpoint (dice={best_dice:.4f})")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= cfg["early_stop_patience"]:
                print(f"  no improvement in {epochs_without_improvement} epochs, stopping early")
                break

    return best_dice




#### Run

In [13]:
print(torch.cuda.is_available())

True


In [14]:
set_seed(CONFIG["seed"])

device = torch.device("cuda")
print(f"using device: {device}")

train(CONFIG, device, resume=True)
# predict(CONFIG, device)

using device: cuda


2026-07-27 01:45:02,202 INFO HTTP Request: HEAD https://huggingface.co/smp-hub/resnet34.imagenet/resolve/7a57b34f723329ff020b3f8bc41771163c519d0c/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-27 01:45:02,210 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/smp-hub/resnet34.imagenet/7a57b34f723329ff020b3f8bc41771163c519d0c/config.json "HTTP/1.1 200 OK"
2026-07-27 01:45:02,218 INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/models/smp-hub/resnet34.imagenet/7a57b34f723329ff020b3f8bc41771163c519d0c/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

2026-07-27 01:45:02,253 INFO HTTP Request: HEAD https://huggingface.co/smp-hub/resnet34.imagenet/resolve/7a57b34f723329ff020b3f8bc41771163c519d0c/model.safetensors "HTTP/1.1 302 Found"
2026-07-27 01:45:02,316 INFO HTTP Request: GET https://huggingface.co/api/models/smp-hub/resnet34.imagenet/xet-read-token/7a57b34f723329ff020b3f8bc41771163c519d0c "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

...

2026-07-27 06:50:44,360 INFO train step 25/1875 (1%) loss=0.7129 elapsed=15s eta=1125s
2026-07-27 06:50:59,572 INFO train step 50/1875 (3%) loss=0.5488 elapsed=30s eta=1110s
2026-07-27 06:51:14,904 INFO train step 78/1875 (4%) loss=0.4982 elapsed=46s eta=1054s
2026-07-27 06:51:30,233 INFO train step 107/1875 (6%) loss=0.4818 elapsed=61s eta=1009s
2026-07-27 06:51:45,420 INFO train step 135/1875 (7%) loss=0.4825 elapsed=76s eta=983s
2026-07-27 06:52:00,932 INFO train step 162/1875 (9%) loss=0.4672 elapsed=92s eta=970s
2026-07-27 06:52:16,379 INFO train step 188/1875 (10%) loss=0.4662 elapsed=107s eta=962s
2026-07-27 06:52:31,895 INFO train step 216/1875 (12%) loss=0.4633 elapsed=123s eta=943s
2026-07-27 06:52:47,145 INFO train step 244/1875 (13%) loss=0.4599 elapsed=138s eta=922s
2026-07-27 06:53:02,492 INFO train step 272/1875 (15%) loss=0.4544 elapsed=153s eta=904s
2026-07-27 06:53:17,816 INFO train step 299/1875 (16%) loss=0.4552 elapsed=169s eta=889s
2026-07-27 06:53:33,293 INFO tra

epoch 13/15 lr=5.00e-05 train_loss=0.4309 val_loss=0.4291 dice=0.3026 (n=107) empty_correct=1.0000 (n=0) instance_f1=0.0153 (precision=0.0102 recall=0.0307)


2026-07-27 07:14:13,717 INFO train step 25/1875 (1%) loss=0.4047 elapsed=16s eta=1157s
2026-07-27 07:14:28,735 INFO train step 50/1875 (3%) loss=0.3791 elapsed=31s eta=1119s
2026-07-27 07:14:43,813 INFO train step 78/1875 (4%) loss=0.3863 elapsed=46s eta=1054s
2026-07-27 07:14:59,112 INFO train step 107/1875 (6%) loss=0.3687 elapsed=61s eta=1008s
2026-07-27 07:15:14,411 INFO train step 135/1875 (7%) loss=0.3788 elapsed=76s eta=984s
2026-07-27 07:15:29,777 INFO train step 161/1875 (9%) loss=0.3753 elapsed=92s eta=976s
2026-07-27 07:15:45,054 INFO train step 187/1875 (10%) loss=0.3664 elapsed=107s eta=966s
2026-07-27 07:16:00,366 INFO train step 215/1875 (11%) loss=0.3632 elapsed=122s eta=944s
2026-07-27 07:16:15,447 INFO train step 243/1875 (13%) loss=0.3656 elapsed=137s eta=923s
2026-07-27 07:16:30,896 INFO train step 271/1875 (14%) loss=0.3550 elapsed=153s eta=904s
2026-07-27 07:16:46,479 INFO train step 298/1875 (16%) loss=0.3563 elapsed=168s eta=891s
2026-07-27 07:17:02,020 INFO tra

epoch 14/15 lr=5.00e-05 train_loss=0.3706 val_loss=0.4307 dice=0.3343 (n=107) empty_correct=1.0000 (n=0) instance_f1=0.0282 (precision=0.0205 recall=0.0455)


0.38133164744928993

In [15]:
# def predict(cfg, device):
#     os.makedirs(os.path.dirname(cfg["submission_path"]), exist_ok=True)

#     test_files = list_image_files(cfg["test_images_dir"])
#     test_dataset = SolarFilamentDataset(
#         test_files,
#         images_dir=cfg["test_images_dir"],
#         transform=get_val_transform(),
#         is_test=True,
#         tile_size=cfg["tile_size"],
#         overlap=cfg["overlap"],
#         image_height=cfg["image_height"],
#         image_width=cfg["image_width"],
#     )
#     test_loader = DataLoader(
#         test_dataset, batch_size=cfg["batch_size"], shuffle=False,
#         num_workers=cfg["num_workers"], pin_memory=True,
#     )

#     model = load_model(cfg["checkpoint_path"], device)
#     results = predict_tiles(model, test_loader, device)
#     build_submission(
#         results,
#         threshold=cfg["pred_threshold"],
#         min_area=cfg["min_area"],
#         output_csv=cfg["submission_path"],
#         tile_size=cfg["tile_size"],
#         image_height=cfg["image_height"],
#         image_width=cfg["image_width"],
#     )